# 🎙️ TTS Motor Karşılaştırma — Bitirme Projesi

Bu notebook 3 farklı TTS motorunu aynı Türkçe cümlelerle test eder:
1. **Coqui XTTS v2** (mevcut motor - optimize parametrelerle)
2. **Chatterbox** (Resemble AI - duygu kontrolü ile)
3. **Fish Speech** (DualAR transformer)

---
**Başlamadan önce:**
- Runtime → Change runtime type → **GPU (T4)** seçin
- Google Drive'a `BITIRME/data/reference_voice/` klasörünü ve `BITIRME/scripts/tts/postprocess.py` dosyasını yükleyin

## Hücre 1 — Google Drive Bağlantısı & GPU Kontrolü

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# === Drive'daki proje konumunuzu buraya yazın ===
PROJECT_DIR = "/content/drive/MyDrive/BITIRME"

# Referans sesleri kontrol et
ref_dir = os.path.join(PROJECT_DIR, "data/reference_voice")
if os.path.exists(ref_dir):
    files = os.listdir(ref_dir)
    print(f"✅ Referans ses klasörü bulundu: {len(files)} dosya")
    for f in sorted(files):
        size_kb = os.path.getsize(os.path.join(ref_dir, f)) / 1024
        print(f"   📁 {f} ({size_kb:.0f} KB)")
else:
    print(f"❌ Klasör bulunamadı: {ref_dir}")
    print("Drive'daki doğru yolu PROJECT_DIR'e yazın!")

# GPU kontrolü
print("\n" + "="*50)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## Hücre 2 — Ortak Bağımlılıklar

In [ ]:
!pip install -q soundfile scipy numpy

## Hücre 3 — Test Cümleleri & Referans Ses Ayarları

In [ ]:
import os
import time
import numpy as np
import soundfile as sf
from IPython.display import Audio, display

# Test cümleleri — farklı duygu/durumlar
TEST_SENTENCES = {
    "neutral": "Merhaba, ben bankadan arıyorum. Hesabınızla ilgili bir bilgilendirme yapmak istiyorum.",
    "urgent": "Dikkat! Hesabınızda şüpheli bir işlem tespit ettik. Hemen doğrulama yapmanız gerekiyor.",
    "friendly": "İyi günler, size özel bir kampanyamız var. Birkaç dakikanızı alabilir miyim?",
    "worried": "Bir dakika, bu biraz garip geldi. Gerçekten bankadan mı arıyorsunuz?",
    "threatening": "Eğer beş dakika içinde bilgilerinizi doğrulamazsanız, hesabınız kalıcı olarak kapatılacak.",
}

# Referans ses dosyaları
AGENT_REF = os.path.join(PROJECT_DIR, "data/reference_voice/agent.wav")
VICTIM_REF = os.path.join(PROJECT_DIR, "data/reference_voice/victim.wav")

# Çıktı dizini
OUTPUT_DIR = "/content/tts_comparison_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Kontrol
for name, path in [("Agent", AGENT_REF), ("Victim", VICTIM_REF)]:
    if os.path.exists(path):
        info = sf.info(path)
        print(f"✅ {name}: {info.duration:.1f}s, {info.samplerate}Hz")
    else:
        print(f"❌ {name}: BULUNAMADI — {path}")

print(f"\n📝 {len(TEST_SENTENCES)} test cümlesi hazır")
print(f"📁 Çıktı: {OUTPUT_DIR}")

## Hücre 4 — Post-Processing Fonksiyonu
Robotsu sesi iyileştiren filtreler. Drive'daki `postprocess.py` dosyasını kullanır veya burada inline tanımlar.

In [ ]:
from scipy.signal import butter, sosfilt

def highpass(audio, sr, cutoff=80.0):
    sos = butter(4, cutoff, btype='high', fs=sr, output='sos')
    return sosfilt(sos, audio).astype(np.float32)

def warmth_eq(audio, sr, gain_db=2.5):
    sos = butter(2, [200, 400], btype='band', fs=sr, output='sos')
    band = sosfilt(sos, audio)
    gain = 10.0 ** (gain_db / 20.0)
    return (audio + band * (gain - 1.0)).astype(np.float32)

def demetallic(audio, sr, cut_db=-3.0):
    sos = butter(2, [3000, 5000], btype='band', fs=sr, output='sos')
    band = sosfilt(sos, audio)
    gain = 10.0 ** (cut_db / 20.0)
    return (audio - band * (1.0 - gain)).astype(np.float32)

def normalize_peak(audio, target_db=-1.0):
    peak = np.max(np.abs(audio))
    if peak < 1e-8:
        return audio
    target_amp = 10.0 ** (target_db / 20.0)
    return (audio * (target_amp / peak)).astype(np.float32)

def soft_compress(audio, threshold_db=-18.0, ratio=3.0):
    threshold = 10.0 ** (threshold_db / 20.0)
    out = audio.copy()
    mask = np.abs(out) > threshold
    above = np.abs(out[mask])
    compressed = threshold + (above - threshold) / ratio
    out[mask] = np.sign(out[mask]) * compressed
    return out.astype(np.float32)

def postprocess(audio, sr):
    """5 aşamalı ses iyileştirme pipeline."""
    audio = highpass(audio, sr)
    audio = warmth_eq(audio, sr)
    audio = demetallic(audio, sr)
    audio = soft_compress(audio)
    audio = normalize_peak(audio)
    return audio

def postprocess_file(input_path, output_path):
    audio, sr = sf.read(input_path, dtype='float32')
    processed = postprocess(audio, sr)
    sf.write(output_path, processed, sr)

print("✅ Post-processing fonksiyonları hazır")

---
# 🔊 Motor 1: Coqui XTTS v2 (Optimize Edilmiş)

## Hücre 5 — Coqui XTTS Kurulum

In [ ]:
!pip install -q coqui-tts

## Hücre 6 — Coqui XTTS Sentez

In [ ]:
import os
os.environ["COQUI_TOS_AGREED"] = "1"

from TTS.api import TTS
import torch

print("Model yükleniyor...")
device = "cuda" if torch.cuda.is_available() else "cpu"
coqui_model = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
print(f"✅ Coqui XTTS v2 yüklendi ({device})")

# ÖNCEKİ parametreler: sadece {"temperature": 0.85}
# YENİ optimize parametreler:
COQUI_KWARGS = {
    "temperature": 0.72,
    "repetition_penalty": 5.0,
    "top_k": 50,
    "top_p": 0.85,
    "speed": 0.95,
}

coqui_results = {}

for emotion, text in TEST_SENTENCES.items():
    ref = VICTIM_REF if emotion == "worried" else AGENT_REF
    out_path = os.path.join(OUTPUT_DIR, f"coqui_{emotion}.wav")

    print(f"\n🎭 [{emotion}] {text[:60]}...")
    start = time.time()

    coqui_model.tts_to_file(
        text=text,
        file_path=out_path,
        speaker_wav=ref,
        language="tr",
        split_sentences=True,
        **COQUI_KWARGS,
    )

    elapsed = time.time() - start
    info = sf.info(out_path)
    coqui_results[emotion] = {"path": out_path, "duration": info.duration, "time": elapsed}

    # Post-process
    pp_path = os.path.join(OUTPUT_DIR, f"coqui_{emotion}_pp.wav")
    postprocess_file(out_path, pp_path)
    coqui_results[emotion]["pp_path"] = pp_path

    print(f"   ✅ {info.duration:.1f}s ses — {elapsed:.1f}s sürdü")
    print(f"   🔊 Ham:")
    display(Audio(out_path))
    print(f"   🔊 Post-processed:")
    display(Audio(pp_path))

print("\n" + "="*50)
print("✅ Coqui XTTS v2 tamamlandı!")

---
# 🔊 Motor 2: Chatterbox (Resemble AI)

## Hücre 7 — Chatterbox Kurulum

In [ ]:
!pip install -q chatterbox-tts

## Hücre 8 — Chatterbox Sentez

In [ ]:
from chatterbox.tts import ChatterboxTTS
import torchaudio
import torch

print("Model yükleniyor...")
device = "cuda" if torch.cuda.is_available() else "cpu"
cb_model = ChatterboxTTS.from_pretrained(device=device)
print(f"✅ Chatterbox yüklendi ({device})")

# Duygu → exaggeration eşlemesi
# Yüksek değer = daha yoğun duygu ifadesi
EMOTION_EXG = {
    "neutral":     0.3,
    "friendly":    0.5,
    "worried":     0.6,
    "urgent":      0.7,
    "threatening": 0.8,
}

cb_results = {}

for emotion, text in TEST_SENTENCES.items():
    ref = VICTIM_REF if emotion == "worried" else AGENT_REF
    out_path = os.path.join(OUTPUT_DIR, f"chatterbox_{emotion}.wav")

    print(f"\n🎭 [{emotion}] (exaggeration={EMOTION_EXG[emotion]}) {text[:50]}...")
    start = time.time()

    wav = cb_model.generate(
        text=text,
        audio_prompt_path=ref,
        exaggeration=EMOTION_EXG[emotion],
    )

    elapsed = time.time() - start

    if wav.dim() == 1:
        wav = wav.unsqueeze(0)
    torchaudio.save(out_path, wav.cpu(), cb_model.sr)

    info = sf.info(out_path)
    cb_results[emotion] = {"path": out_path, "duration": info.duration, "time": elapsed}

    # Post-process
    pp_path = os.path.join(OUTPUT_DIR, f"chatterbox_{emotion}_pp.wav")
    postprocess_file(out_path, pp_path)
    cb_results[emotion]["pp_path"] = pp_path

    print(f"   ✅ {info.duration:.1f}s ses — {elapsed:.1f}s sürdü")
    print(f"   🔊 Ham:")
    display(Audio(out_path))
    print(f"   🔊 Post-processed:")
    display(Audio(pp_path))

print("\n" + "="*50)
print("✅ Chatterbox tamamlandı!")

---
# 🔊 Motor 3: Fish Speech

## Hücre 9 — Fish Speech Kurulum

In [ ]:
!pip install -q fish-speech

## Hücre 10 — Fish Speech Sentez
Fish Speech API'si sürüme göre değişebilir. Hata alırsanız bir sonraki hücreye geçin.

In [ ]:
fish_results = {}

try:
    # Yöntem 1: Python API
    from fish_speech.models import TTSModel

    print("Model yükleniyor...")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    fish_model = TTSModel.from_pretrained(device=device)
    print(f"✅ Fish Speech yüklendi ({device})")

    for emotion, text in TEST_SENTENCES.items():
        ref = VICTIM_REF if emotion == "worried" else AGENT_REF
        out_path = os.path.join(OUTPUT_DIR, f"fish_{emotion}.wav")

        print(f"\n🎭 [{emotion}] {text[:50]}...")
        start = time.time()

        result = fish_model.synthesize(
            text=text,
            reference_audio=ref,
            language="tr",
        )

        elapsed = time.time() - start
        audio = np.array(result.audio, dtype=np.float32)
        sr = getattr(result, 'sample_rate', 44100)
        sf.write(out_path, audio, sr)

        info = sf.info(out_path)
        fish_results[emotion] = {"path": out_path, "duration": info.duration, "time": elapsed}

        # Post-process
        pp_path = os.path.join(OUTPUT_DIR, f"fish_{emotion}_pp.wav")
        postprocess_file(out_path, pp_path)
        fish_results[emotion]["pp_path"] = pp_path

        print(f"   ✅ {info.duration:.1f}s ses — {elapsed:.1f}s sürdü")
        print(f"   🔊 Ham:")
        display(Audio(out_path))
        print(f"   🔊 Post-processed:")
        display(Audio(pp_path))

    print("\n✅ Fish Speech tamamlandı!")

except ImportError as e:
    print(f"⚠️  Fish Speech import hatası: {e}")
    print("\nAlternatif kurulum deneyin:")
    print("  !git clone https://github.com/fishaudio/fish-speech.git")
    print("  %cd fish-speech")
    print("  !pip install -e .")
except Exception as e:
    print(f"⚠️  Fish Speech hatası: {e}")
    print("Bu motor atlanacak, diğer 2 motorla devam edebilirsiniz.")

---
# 📊 Karşılaştırma & Değerlendirme

## Hücre 11 — Yan Yana Dinleme

In [ ]:
all_engines = [
    ("🟠 Coqui XTTS v2", coqui_results),
    ("🟢 Chatterbox",     cb_results),
    ("🔵 Fish Speech",    fish_results),
]

for emotion, text in TEST_SENTENCES.items():
    print("\n" + "="*60)
    print(f"🎭 DUYGU: {emotion.upper()}")
    print(f"📝 \"{text}\"")
    print("="*60)

    for engine_name, results in all_engines:
        if emotion not in results:
            print(f"\n  {engine_name}: ❌ atlandı")
            continue

        r = results[emotion]
        print(f"\n  {engine_name} — {r['duration']:.1f}s ses, {r['time']:.1f}s inference")
        print(f"  Ham:")
        display(Audio(r["path"]))
        if "pp_path" in r:
            print(f"  Post-processed:")
            display(Audio(r["pp_path"]))

## Hücre 12 — Özet Tablo

In [ ]:
print("\n" + "="*65)
print("ÖZET TABLO")
print("="*65)
print(f"{'Motor':<22} {'Ort. Inference (s)':<22} {'Ort. Ses Süresi (s)':<22}")
print("-"*65)

for engine_name, results in all_engines:
    if not results:
        print(f"{engine_name:<22} {'— atlandı':}")
        continue
    avg_time = np.mean([r['time'] for r in results.values()])
    avg_dur  = np.mean([r['duration'] for r in results.values()])
    print(f"{engine_name:<22} {avg_time:<22.1f} {avg_dur:<22.1f}")

print()
print("📋 Sonraki adımlar:")
print("  1. Yukarıdaki sesleri dinleyin, en doğal olanı seçin")
print("  2. Seçtiğiniz motoru projedeki pipeline'a entegre edin")
print("  3. Tüm diyalog planlarını seçilen motorla sentezleyin")

## Hücre 13 — Sonuçları Drive'a Kaydet

In [ ]:
import json

# Sonuçları JSON olarak kaydet
comparison = {
    "test_sentences": TEST_SENTENCES,
    "engines": {},
}

for engine_key, results in [("coqui_xtts_v2", coqui_results),
                             ("chatterbox", cb_results),
                             ("fish_speech", fish_results)]:
    if results:
        comparison["engines"][engine_key] = {
            emotion: {
                "duration_s": round(r["duration"], 2),
                "inference_s": round(r["time"], 2),
            }
            for emotion, r in results.items()
        }

# Drive'a kaydet
save_dir = os.path.join(PROJECT_DIR, "outputs/tts_engine_comparison")
os.makedirs(save_dir, exist_ok=True)

result_path = os.path.join(save_dir, "comparison_results.json")
with open(result_path, 'w', encoding='utf-8') as f:
    json.dump(comparison, f, ensure_ascii=False, indent=2)
print(f"✅ Sonuçlar kaydedildi: {result_path}")

# Ses dosyalarını da Drive'a kopyala
import shutil
for fname in os.listdir(OUTPUT_DIR):
    if fname.endswith('.wav'):
        src = os.path.join(OUTPUT_DIR, fname)
        dst = os.path.join(save_dir, fname)
        shutil.copy2(src, dst)

wav_count = len([f for f in os.listdir(save_dir) if f.endswith('.wav')])
print(f"✅ {wav_count} ses dosyası Drive'a kopyalandı: {save_dir}")